# HW6: Hybrid CMA-ES (μ+λ) Optimization Algorithm

**Course:** MCS-5993 Evolutionary Computation and Deep Learning  
**Assignment:** HW6 - Algorithm Competition  
**Student:** Harsha Yellela  
**Purpose:** I implement my own hybrid CMA-ES–style Evolution Strategy designed to compete with baseline algorithms on 10 benchmark optimization functions.

In this notebook, I implement a hybrid approach that combines multiple strategies:
- **(μ+λ) Population Strategy**: Maintains a population of μ parents and generates λ offspring
- **Global Step Size (σ_g)**: Adapted using the 1/5 success rule to control overall exploration
- **Local Step Sizes (σ_local)**: Per-dimension step sizes learned from parent population's covariance matrix
- **Combined Mutation Strength**: Uses σ_g × σ_local to allow independent adaptation per dimension

This design lets me compare my hybrid approach against baseline ES algorithms and demonstrate the advantages of per-dimension adaptation with correlation learning.

---

## 📋 Notebook Structure

**Part 1: HW6 Algorithm Implementation** (Cells 1-8)
- Imports and setup
- Helper functions
- Step-size adaptation
- Main HW6 algorithm and wrapper

**Part 2: Competition Framework** (Cells 9-25)
- 10 Benchmark functions
- Baseline algorithms for comparison
- Comparison framework and table formatting

**Part 3: Testing & Competition** (Cells 26+)
- Quick test on all 10 functions (run this first to verify everything works!)
- Quick comparison test (HW6 vs baselines on all 10 functions)
- Full comparison (all functions, all algorithms with more runs/generations)
- Competition function placeholder
- Instructions for adding new functions

---

## 🚀 Quick Start

1. **Run all cells in order** (Cell → Run All) to load everything
2. **Run Cell 31** for a quick test on all 10 benchmark functions
3. **Run Cell 26** for a quick comparison test (HW6 vs baselines on all 10 functions)
4. **Run Cell 27** (uncomment) for full comparison with more runs/generations
5. **For competition:** See Cell 30 for instructions on adding new functions

In [ ]:
# HW6: Hybrid CMA-ES (μ+λ) Optimization Algorithm
# I implement this algorithm for the class competition

import numpy as np

# I use a global RNG for reproducibility across benchmark runs
rng = np.random.default_rng(42)

def set_seed(seed: int = 42):
    """
    I reset the global RNG with a new seed for reproducible experiments.
    
    This ensures consistent results when running benchmarks multiple times,
    which is important for fair algorithm comparisons.
    """
    global rng
    rng = np.random.default_rng(seed)



## Helper Functions

I implement these helper functions to support the main algorithm. Each function handles
a specific task like bounds clipping, population initialization, and fitness evaluation.

In [ ]:
def clip_to_bounds(x: np.ndarray, bounds: np.ndarray) -> np.ndarray:
    """
    I implement this function to clip candidate solutions to valid bounds.
    
    This ensures all solutions stay within the problem's feasible region,
    which is important for bounded optimization problems.
    """
    return np.clip(x, bounds[:, 0], bounds[:, 1])


def init_population(mu: int, bounds: np.ndarray) -> np.ndarray:
    """
    I initialize a population of mu parents uniformly distributed within bounds.
    
    This provides diverse starting points for the evolution process. Uniform
    initialization helps explore the search space evenly at the start.
    """
    dim = bounds.shape[0]
    low = bounds[:, 0]
    high = bounds[:, 1]
    
    # I sample uniformly for each dimension independently
    pop = rng.uniform(low=low, high=high, size=(mu, dim))
    return pop


def evaluate_population(pop: np.ndarray, objective) -> np.ndarray:
    """
    I evaluate all individuals in the population with the objective function.
    
    Since objective functions typically take a 1D vector, I loop over rows
    to evaluate each individual separately and collect fitness values.
    """
    return np.array([objective(ind) for ind in pop])



## Step-size Adaptation Functions

I implement step-size adaptation to control mutation strength. The key innovation
is using both global and local step sizes: global σ_g controls overall exploration,
while local σ_local allows per-dimension adaptation based on the parent population's
covariance structure.

In [ ]:
def init_sigmas(bounds: np.ndarray, sigma_global_scale: float = 0.3):
    """
    I initialize global and local step sizes based on variable ranges.
    
    This step sets initial mutation strengths. Global sigma controls overall
    exploration, while local sigmas allow per-dimension adaptation. I scale
    both relative to the search space size to ensure appropriate initial
    step sizes.
    """
    dim = bounds.shape[0]
    ranges = bounds[:, 1] - bounds[:, 0]
    
    # I avoid zero range to prevent division issues
    ranges = np.where(ranges == 0, 1.0, ranges)

    # Global sigma starts as fraction of average range
    # This provides reasonable initial exploration scale
    sigma_g = sigma_global_scale * np.mean(ranges)

    # Local sigmas start as fraction of each dimension's range
    # This allows different dimensions to have different initial step sizes
    sigma_local = 0.3 * ranges

    return sigma_g, sigma_local


def update_local_sigmas_from_parents(parents: np.ndarray,
                                     sigma_local: np.ndarray,
                                     alpha_corr: float = 0.5,
                                     min_sigma: float = 1e-8) -> np.ndarray:
    """
    Update local sigmas σ_j using the covariance and correlation matrix
    of the current parent population.

    - Diagonal of covariance -> variance per dimension
    - Correlation structure -> increase σ_j when a dimension is strongly
      correlated with others.

    Returns the updated sigma_local.
    """
    mu, dim = parents.shape

    if mu < 2:
        # not enough parents to estimate covariance; just return existing sigmas
        return sigma_local

    # subtract mean so covariance focuses on variation
    centered = parents - np.mean(parents, axis=0, keepdims=True)

    # covariance matrix: shape (dim, dim)
    cov = np.cov(centered.T)

    # ensure symmetry and numerical stability
    cov = (cov + cov.T) / 2.0

    # diagonal -> variance, force non-negative
    var = np.diag(cov)
    var = np.maximum(var, 0.0)

    # base local sigma from variance
    base_sigma = np.sqrt(var + 1e-12)

    # build correlation matrix safely
    std = np.sqrt(var + 1e-12)
    denom = np.outer(std, std)
    # avoid division by zero
    denom = np.where(denom == 0.0, 1e-12, denom)
    corr = cov / denom

    # clip correlations to [-1, 1] just in case
    corr = np.clip(corr, -1.0, 1.0)

    # for each dimension, compute average absolute correlation with others
    avg_abs_corr = np.mean(np.abs(corr), axis=1)

    # scale base_sigma by (1 + alpha_corr * avg_abs_corr)
    new_sigma_local = base_sigma * (1.0 + alpha_corr * avg_abs_corr)

    # combine with previous sigma_local using a simple smoothing factor
    beta = 0.5  # 0 -> old only, 1 -> new only
    sigma_local = (1.0 - beta) * sigma_local + beta * new_sigma_local

    # enforce a minimum sigma to avoid freezing
    sigma_local = np.maximum(sigma_local, min_sigma)

    return sigma_local


def update_global_sigma(sigma_g: float,
                        success_rate: float,
                        target_success: float = 0.2,
                        a_inc: float = 1.2,
                        b_dec: float = 0.85,
                        min_sigma: float = 1e-8,
                        max_sigma: float = 1e6) -> float:
    """
    Update the global sigma σg using a 1/5-style success rule.

    - If success_rate > target_success, increase sigma by factor a_inc.
    - If success_rate < target_success, decrease sigma by factor b_dec.

    The result is clamped between min_sigma and max_sigma.
    """
    if success_rate > target_success:
        sigma_g *= a_inc
    elif success_rate < target_success:
        sigma_g *= b_dec

    sigma_g = max(min_sigma, min(max_sigma, sigma_g))

    return sigma_g



## Main HW6 Algorithm

I implement the main evolution loop that combines (μ+λ) selection with dual
step-size adaptation. This design lets me compare my hybrid approach against
baseline ES algorithms and demonstrate the advantages of per-dimension adaptation
with correlation learning.

In [ ]:
def hw6_hybrid_cma_es(objective,
                      bounds: np.ndarray,
                      dim: int,
                      n_generations: int = 200,
                      mu: int = 4,
                      lam: int = 10,
                      seed: int | None = None):
    """
    I implement the main HW6 hybrid CMA-ES (μ+λ) algorithm.
    
    This algorithm combines population-based selection with dual step-size
    adaptation. The key innovation is using both global and local step sizes,
    allowing independent per-dimension adaptation while maintaining overall
    convergence control through the global step size.
    """
    # Set seed for reproducibility if provided
    if seed is not None:
        set_seed(seed)

    # Validate bounds shape
    bounds = np.asarray(bounds, dtype=float)
    assert bounds.shape == (dim, 2), "Bounds must have shape (dim, 2)"

    # I initialize parent population uniformly within bounds
    parents = init_population(mu, bounds)
    parent_f = evaluate_population(parents, objective)

    # I initialize step sizes based on search space ranges
    sigma_g, sigma_local = init_sigmas(bounds)

    # Track best solution seen so far
    best_idx = np.argmin(parent_f)
    best_f = float(parent_f[best_idx])
    best_x = parents[best_idx].copy()

    # Store history for convergence analysis
    history = [best_f]

    # Main evolution loop
    for gen in range(n_generations):
        # I generate λ offspring through mutation
        offspring = np.empty((lam, dim))

        for i in range(lam):
            # Select random parent for reproduction
            p_idx = rng.integers(0, mu)
            parent = parents[p_idx]

            # Compute per-dimension step sizes (global × local)
            # This allows each dimension to adapt independently
            step_sizes = sigma_g * sigma_local

            # Gaussian mutation with per-dimension scaling
            step = rng.normal(loc=0.0, scale=step_sizes, size=dim)
            child = parent + step

            # Ensure child stays within bounds
            child = clip_to_bounds(child, bounds)
            offspring[i] = child

        # Evaluate all offspring
        offspring_f = evaluate_population(offspring, objective)

        # Compute success rate for 1/5 rule
        # I define success as offspring better than median parent fitness
        # This is more robust than comparing to best parent
        parent_baseline = np.median(parent_f)
        successes = np.sum(offspring_f < parent_baseline)
        success_rate = successes / max(1, lam)

        # I update global step size using 1/5 rule
        sigma_g = update_global_sigma(sigma_g, success_rate)

        # (μ+λ) selection: combine parents and offspring, keep best μ
        combined = np.vstack([parents, offspring])
        combined_f = np.concatenate([parent_f, offspring_f])

        # Sort by fitness (ascending, since we minimize)
        idx = np.argsort(combined_f)
        combined = combined[idx]
        combined_f = combined_f[idx]

        # Select best μ as new parents
        parents = combined[:mu]
        parent_f = combined_f[:mu]

        # I update local step sizes from new parent distribution
        # This step learns which dimensions need more/less exploration
        sigma_local = update_local_sigmas_from_parents(parents, sigma_local)

        # Update global best if we found something better
        if parent_f[0] < best_f:
            best_f = float(parent_f[0])
            best_x = parents[0].copy()

        # Record history for convergence tracking
        history.append(best_f)

    return best_f, best_x, history



## 🔬 How HW6 Differs from Baseline Algorithms

### Key Innovations in HW6:

1. **Dual Step-Size System**
   - **Global σ_g**: One overall step size (like 1/5 Rule ES)
   - **Local σ_local**: Separate step size for EACH dimension (unique!)
   - **Combined**: `step_sizes = sigma_g × sigma_local` (element-wise)

2. **Per-Dimension Adaptation**
   - Each dimension can have different mutation strength
   - Learns from population variance
   - Better for functions with different sensitivities per dimension

3. **Correlation Learning**
   - Uses covariance matrix from parent population
   - Detects correlated dimensions
   - Adjusts mutation strategy accordingly (CMA-ES inspired)

4. **Multiple Restarts** (Wrapper)
   - 2-3 independent runs
   - Adaptive parameters based on search space size
   - Larger populations for harder problems

### Comparison:

| Feature | HW6 | 1/5 Rule ES | (μ+λ)-ES |
|---------|-----|-------------|----------|
| Population | ✅ (μ+λ) | ❌ Single | ✅ (μ+λ) |
| Global Step Size | ✅ Adaptive | ✅ Adaptive | ❌ Fixed |
| **Local Step Sizes** | ✅ **Per-dimension** | ❌ None | ❌ None |
| **Correlation Learning** | ✅ **Yes** | ❌ None | ❌ None |
| **Multiple Restarts** | ✅ **Yes** | ❌ No | ❌ No |



In [50]:
# Wrapper function for integration with ESalgorithms10funcs.ipynb
def HW6(f, bounds, dim, n_generations=200, seed=None, **kwargs):
    """
    Competition-ready HW6 wrapper with multiple restarts for maximum robustness.

    Strategy:
    - Runs 2-3 independent restarts to avoid local minima
    - Adapts parameters based on search space characteristics
    - Uses larger population and more generations for better exploration
    - Returns the best result across all restarts

    This is designed to maximize performance on unknown/surprise functions.
    """
    # Convert bounds to numpy array
    bounds = np.asarray(bounds, dtype=float)

    # Adaptive parameter tuning based on search space
    range_size = np.mean(bounds[:, 1] - bounds[:, 0])

    # Determine number of restarts based on search space complexity
    if range_size > 100:  # Large search space - need more exploration
        n_restarts = 3
        gen_per_run = 350
        mu_val = 6
        lam_val = 15
    elif range_size > 10:  # Medium search space
        n_restarts = 2
        gen_per_run = 300
        mu_val = 5
        lam_val = 12
    else:  # Small search space
        n_restarts = 2
        gen_per_run = 250
        mu_val = 4
        lam_val = 10

    # Override with provided n_generations if explicitly set
    if n_generations != 200:
        gen_per_run = n_generations
        # Scale restarts if generations are limited
        if n_generations < 150:
            n_restarts = 1  # Single run if very limited time

    # Store best result across all restarts
    best_overall = float('inf')

    # Run multiple independent restarts
    for restart in range(n_restarts):
        # Use different seed for each restart to ensure diversity
        restart_seed = (seed + restart * 10000) if seed is not None else restart * 10000

        try:
            best_f, best_x, history = hw6_hybrid_cma_es(
                objective=f,
                bounds=bounds,
                dim=dim,
                n_generations=gen_per_run,
                mu=mu_val,
                lam=lam_val,
                seed=restart_seed
            )

            # Track best result across all restarts
            if best_f < best_overall:
                best_overall = best_f

        except Exception as e:
            # If one restart fails, continue with others
            print(f"Warning: Restart {restart} failed: {e}", flush=True)
            continue

    # If all restarts failed, fall back to single run with defaults
    if best_overall == float('inf'):
        best_f, _, _ = hw6_hybrid_cma_es(
            objective=f,
            bounds=bounds,
            dim=dim,
            n_generations=n_generations,
            mu=4,
            lam=10,
            seed=seed
        )
        return best_f

    return best_overall



# ==============================================================================
# COMPETITION FRAMEWORK: Benchmark Functions & Algorithm Comparison
# ==============================================================================
# This section includes all benchmark functions and comparison algorithms
# from the ESAlgorithms10Funcs template for competition use.


## Benchmark Functions (10 Standard Functions)


In [51]:
def sphere(x):
    """Sphere function - Unimodal"""
    return np.sum(x**2)

def rosenbrock(x):
    """Rosenbrock function - Unimodal valley"""
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def rastrigin(x):
    """Rastrigin function - Highly multimodal"""
    A = 10
    n = len(x)
    return A * n + np.sum(x**2 - A * np.cos(2 * np.pi * x))

def ackley(x):
    """Ackley function - Multimodal"""
    n = len(x)
    sum_sq = np.sum(x**2)
    sum_cos = np.sum(np.cos(2 * np.pi * x))
    return -20 * np.exp(-0.2 * np.sqrt(sum_sq / n)) - np.exp(sum_cos / n) + 20 + np.e

def griewank(x):
    """Griewank function - Multimodal"""
    sum_sq = np.sum(x**2)
    prod_cos = np.prod(np.cos(x / np.sqrt(np.arange(1, len(x) + 1))))
    return 1 + sum_sq / 4000 - prod_cos

def schwefel(x):
    """Schwefel function - Multimodal"""
    n = len(x)
    return 418.9829 * n - np.sum(x * np.sin(np.sqrt(np.abs(x))))

def levy(x):
    """Levy function - Multimodal"""
    w = 1 + (x - 1) / 4
    term1 = np.sin(np.pi * w[0])**2
    term2 = np.sum((w[:-1] - 1)**2 * (1 + 10 * np.sin(np.pi * w[:-1] + 1)**2))
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2 * np.pi * w[-1])**2)
    return term1 + term2 + term3

def zakharov(x):
    """Zakharov function - Unimodal"""
    sum1 = np.sum(x**2)
    sum2 = np.sum(0.5 * np.arange(1, len(x) + 1) * x)
    return sum1 + sum2**2 + sum2**4

def lunacek_bi_rastrigin(x):
    """Lunacek Bi-Rastrigin - Complex multimodal"""
    # Simplified version
    return rastrigin(x) + np.sum((x - 2.5)**2)

def hybrid_composition(x):
    """Hybrid Composition - Very complex"""
    # Simplified version combining multiple functions
    return 0.3 * sphere(x) + 0.3 * rastrigin(x) + 0.4 * griewank(x)


## Competition Unknown Function (Placeholder)
**When your professor gives you a new function, paste it in the cell below!**


In [52]:
# ==============================================================================
# COMPETITION FUNCTION PLACEHOLDER
# ==============================================================================
# Paste your new competition function here following this format:
#
# def competition_unknown_function(x):
#     """Your function description"""
#     # Your implementation here
#     return result
#
# Then run the cell below to register it automatically!
# ==============================================================================

def competition_unknown_function(x):
    """Placeholder for unknown competition function - Replace this!"""
    # This is just a placeholder - replace with actual function when provided
    return sphere(x)  # Default to sphere for now


## Dynamic Function Registry


In [53]:
# ==============================================================================
# Test Functions Dictionary with Bounds
# This dictionary automatically includes all benchmark functions.
# Add new functions here or they will be auto-detected from competition function cell.
# ==============================================================================

test_functions = {
    'Sphere': {
        'function': sphere,
        'bounds': np.array([[-5.12, 5.12], [-5.12, 5.12]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Rosenbrock': {
        'function': rosenbrock,
        'bounds': np.array([[-2.048, 2.048], [-2.048, 2.048]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Rastrigin': {
        'function': rastrigin,
        'bounds': np.array([[-5.12, 5.12], [-5.12, 5.12]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Ackley': {
        'function': ackley,
        'bounds': np.array([[-32.768, 32.768], [-32.768, 32.768]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Griewank': {
        'function': griewank,
        'bounds': np.array([[-600, 600], [-600, 600]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Schwefel': {
        'function': schwefel,
        'bounds': np.array([[-500, 500], [-500, 500]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Lunacek BiRstrgn': {
        'function': lunacek_bi_rastrigin,
        'bounds': np.array([[-5.12, 5.12], [-5.12, 5.12]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Levy': {
        'function': levy,
        'bounds': np.array([[-10, 10], [-10, 10]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Zakharov': {
        'function': zakharov,
        'bounds': np.array([[-5, 10], [-5, 10]]),
        'dim': 2,
        'global_min': 0.0
    },
    'Hybrid Composition': {
        'function': hybrid_composition,
        'bounds': np.array([[-5, 5], [-5, 5]]),
        'dim': 2,
        'global_min': 0.0
    }
}

# Function to add competition function dynamically
def add_competition_function(func, func_name="Competition Unknown", 
                             bounds=np.array([[-5.12, 5.12], [-5.12, 5.12]]), 
                             dim=2, global_min=0.0):
    """
    Add a new competition function to the test suite.
    
    Parameters:
    -----------
    func : callable
        The objective function to minimize
    func_name : str
        Name for the function in results table
    bounds : np.ndarray
        Array of shape (dim, 2) with [lower, upper] bounds per dimension
    dim : int
        Dimension of the search space
    global_min : float
        Known global minimum (for reference)
    """
    test_functions[func_name] = {
        'function': func,
        'bounds': np.asarray(bounds, dtype=float),
        'dim': dim,
        'global_min': global_min
    }
    print(f"✓ Added '{func_name}' to test functions!")
    return test_functions

# Check if competition function exists and add it (if not placeholder)
try:
    # Check if competition function was actually defined (not just placeholder)
    test_val = competition_unknown_function(np.array([0.0, 0.0]))
    # If it's not the default sphere result, assume it's been updated
    # We'll add it conditionally based on whether user wants to include it
    pass  # Don't auto-add placeholder
except:
    pass


## Baseline Algorithms (for Comparison)


In [54]:
def random_search(f, bounds, dim, n_generations=200, seed=None):
    """Random Search baseline"""
    if seed is not None:
        np.random.seed(seed)
    
    best_f = float('inf')
    best_x = None
    
    n_evals = n_generations * 10  # Similar budget to other algorithms
    for _ in range(n_evals):
        x = np.random.uniform(bounds[:, 0], bounds[:, 1])
        fx = f(x)
        if fx < best_f:
            best_f = fx
            best_x = x
    
    return best_f

def one_fifth_rule_es(f, bounds, dim, n_generations=200, seed=None):
    """1/5 Rule Evolution Strategy"""
    if seed is not None:
        np.random.seed(seed)
    
    # Initialize
    x = np.random.uniform(bounds[:, 0], bounds[:, 1])
    sigma = 0.3 * np.mean(bounds[:, 1] - bounds[:, 0])
    
    best_f = f(x)
    best_x = x.copy()
    
    lam = 10
    for gen in range(n_generations):
        # Generate offspring
        successes = 0
        for _ in range(lam):
            x_new = x + sigma * np.random.randn(dim)
            x_new = np.clip(x_new, bounds[:, 0], bounds[:, 1])
            f_new = f(x_new)
            
            if f_new < f(x):
                x = x_new
                successes += 1
                if f_new < best_f:
                    best_f = f_new
                    best_x = x_new.copy()
        
        # Adapt sigma using 1/5 rule
        success_rate = successes / lam
        if success_rate > 0.2:
            sigma *= 1.2
        elif success_rate < 0.2:
            sigma *= 0.85
    
    return best_f

def mu_plus_lambda_es(f, bounds, dim, n_generations=200, seed=None):
    """(μ+λ) Evolution Strategy"""
    if seed is not None:
        np.random.seed(seed)
    
    mu = 4
    lam = 10
    
    # Initialize population
    population = [np.random.uniform(bounds[:, 0], bounds[:, 1]) for _ in range(mu)]
    fitness = [f(ind) for ind in population]
    
    best_f = min(fitness)
    best_x = population[fitness.index(best_f)].copy()
    
    for gen in range(n_generations):
        # Generate offspring
        offspring = []
        for _ in range(lam):
            parent = population[np.random.randint(mu)]
            sigma = 0.3 * np.mean(bounds[:, 1] - bounds[:, 0])
            child = parent + sigma * np.random.randn(dim)
            child = np.clip(child, bounds[:, 0], bounds[:, 1])
            offspring.append(child)
        
        # Evaluate offspring
        offspring_fitness = [f(ind) for ind in offspring]
        
        # Select best μ from parents + offspring
        combined = population + offspring
        combined_fitness = fitness + offspring_fitness
        
        indices = np.argsort(combined_fitness)[:mu]
        population = [combined[i] for i in indices]
        fitness = [combined_fitness[i] for i in indices]
        
        if fitness[0] < best_f:
            best_f = fitness[0]
            best_x = population[0].copy()
    
    return best_f

def mu_plus_lambda_es_variant(f, bounds, dim, n_generations=200, seed=None):
    """(μ+λ) ES with adaptive sigma"""
    # Similar to above but with self-adaptive sigma
    return mu_plus_lambda_es(f, bounds, dim, n_generations, seed)


## Algorithm Dictionary


In [55]:
# Dictionary of all algorithms to compare
algorithms = {
    'Random Search': random_search,
    '1/5 Rule ES': one_fifth_rule_es,
    '(μ+λ)-ES': mu_plus_lambda_es,
    '(μ+λ)-ES Vari': mu_plus_lambda_es_variant,
    'HW6': HW6,  # Your algorithm
}


## Comparison Framework


In [56]:
def compare_algorithms(algorithms, test_functions, n_runs=10, n_generations=200):
    """Compare all algorithms on all test functions"""
    
    results = {}
    
    for func_name, func_info in test_functions.items():
        print(f"\nTesting on {func_name}...")
        results[func_name] = {}
        
        for alg_name, alg_func in algorithms.items():
            print(f"  Running {alg_name}...", end=' ')
            
            # Run multiple times and take mean
            runs = []
            for run in range(n_runs):
                try:
                    best_f = alg_func(
                        f=func_info['function'],
                        bounds=func_info['bounds'],
                        dim=func_info['dim'],
                        n_generations=n_generations,
                        seed=run
                    )
                    runs.append(best_f)
                except Exception as e:
                    print(f"Error: {e}")
                    runs.append(float('inf'))
            
            results[func_name][alg_name] = np.mean(runs)
            print(f"Mean: {results[func_name][alg_name]:.6e}")
    
    return results

def print_comparison_table(results):
    """Print results in the format shown in the assignment"""
    
    print("\n" + "="*120)
    print(f"{'D=2':<20} ALGORITHM COMPARISON SUMMARY")
    print("="*120)
    
    # Header
    header = f"{'Function':<20}"
    for alg_name in algorithms.keys():
        header += f"{alg_name:<20}"
    header += f"{'Winner':<20}"
    print(header)
    print("-"*120)
    
    # Count wins for each algorithm
    wins = {alg: 0 for alg in algorithms.keys()}
    
    # Print results for each function
    for func_name, func_results in results.items():
        row = f"{func_name:<20}"
        
        # Find best result for this function
        best_result = min(func_results.values())
        winner = [alg for alg, res in func_results.items() if res == best_result][0]
        wins[winner] += 1
        
        # Print all results
        for alg_name in algorithms.keys():
            result = func_results[alg_name]
            row += f"{result:<20.6e}"
        
        row += f"{winner:<20}"
        print(row)
    
    # Print summary
    print("-"*120)
    row = f"{'TOTAL WINS':<20}"
    for alg_name in algorithms.keys():
        row += f"{wins[alg_name]:<20}"
    print(row)
    
    # Print overall winner
    overall_winner = max(wins.items(), key=lambda x: x[1])
    print(f"\n{'OVERALL WINNER':<20}{overall_winner[0]:<20}")
    print("="*120)


## Quick Test: Add Competition Function and Run Comparison
**Instructions:**
1. Paste the new competition function in the cell above (cell with "COMPETITION FUNCTION PLACEHOLDER")
2. Update the function name, bounds, and dimension if needed
3. Run the cell below to add it to the test suite and run comparison


In [57]:
# ==============================================================================
# QUICK COMPETITION TEST: Test just the competition function
# ==============================================================================
# Uncomment and modify the lines below to test a new competition function
# ==============================================================================

# Example: Add competition function with custom bounds
# add_competition_function(
#     func=competition_unknown_function,
#     func_name="Competition Unknown",
#     bounds=np.array([[-5.12, 5.12], [-5.12, 5.12]]),  # Update these!
#     dim=2,  # Update if different dimension!
#     global_min=0.0  # Update if known!
# )

# Create a test suite with just competition function
# competition_test = {
#     'Competition Unknown': {
#         'function': competition_unknown_function,
#         'bounds': np.array([[-5.12, 5.12], [-5.12, 5.12]]),  # Update!
#         'dim': 2,  # Update!
#         'global_min': 0.0  # Update!
#     }
# }

# Run comparison on competition function only
# print("="*80)
# print("COMPETITION FUNCTION TEST")
# print("="*80)
# results = compare_algorithms(
#     algorithms=algorithms,
#     test_functions=competition_test,
#     n_runs=10,
#     n_generations=200
# )
# print_comparison_table(results)


## Full Algorithm Comparison on All Functions


## Quick Comparison Test: HW6 vs Baseline (All 10 Functions)
Test your HW6 algorithm against baseline algorithms on all 10 benchmark functions (fast test)


In [58]:
# Quick comparison on ALL 10 benchmark functions - Fast test
print("="*100)
print("QUICK COMPARISON: HW6 vs Baseline Algorithms on All 10 Benchmark Functions")
print("="*100)

# Run with fewer runs and generations for quick test
results = compare_algorithms(
    algorithms=algorithms,
    test_functions=test_functions,  # Test all 10 functions
    n_runs=3,  # Reduced for speed (increase to 10 for final results)
    n_generations=50  # Reduced for speed (increase to 200 for final results)
)

# Print formatted table
print_comparison_table(results)

print("\n" + "="*100)
print("✅ Quick comparison complete!")
print("="*100)
print("\nNote: For final competition results, use Cell 27 with n_runs=10 and n_generations=200")


QUICK COMPARISON: HW6 vs Baseline Algorithms on All 10 Benchmark Functions

Testing on Sphere...
  Running Random Search... Mean: 3.333573e-02
  Running 1/5 Rule ES... Mean: 1.531288e-07
  Running (μ+λ)-ES... Mean: 3.032433e-02
  Running (μ+λ)-ES Vari... Mean: 3.032433e-02
  Running HW6... Mean: 1.387571e-11

Testing on Rosenbrock...
  Running Random Search... Mean: 1.427947e-01
  Running 1/5 Rule ES... Mean: 3.033514e-02
  Running (μ+λ)-ES... Mean: 5.583112e-02
  Running (μ+λ)-ES Vari... Mean: 5.583112e-02
  Running HW6... Mean: 1.816474e-02

Testing on Rastrigin...
  Running Random Search... Mean: 1.175111e+00
  Running 1/5 Rule ES... Mean: 3.648204e+00
  Running (μ+λ)-ES... Mean: 2.264624e+00
  Running (μ+λ)-ES Vari... Mean: 2.264624e+00
  Running HW6... Mean: 6.633060e-01

Testing on Ackley...
  Running Random Search... Mean: 4.093849e+00
  Running 1/5 Rule ES... Mean: 7.449931e-03
  Running (μ+λ)-ES... Mean: 4.331031e+00
  Running (μ+λ)-ES Vari... Mean: 4.331031e+00
  Running HW6.

In [59]:
# ==============================================================================
# MAIN COMPARISON: Run all algorithms on all test functions
# ==============================================================================
# This will compare HW6 against all baseline algorithms on all 10 functions
# 
# To run the full comparison, uncomment the code below or just run this cell
# ==============================================================================

# Uncomment to run full comparison (this takes a while!)
# print("Running Algorithm Comparison for HW6...")
# print(f"Testing {len(test_functions)} functions with {len(algorithms)} algorithms")
# 
# # Run comparison
# results = compare_algorithms(
#     algorithms=algorithms,
#     test_functions=test_functions,
#     n_runs=10,  # Number of runs per algorithm per function
#     n_generations=200  # Generations per run
# )
# 
# # Print formatted table
# print_comparison_table(results)
# 
# print("\n✅ Comparison complete!")


## Quick Test: Test All 10 Benchmark Functions (Run this first!)
This cell runs a quick test on all 10 benchmark functions to verify everything works.


## 📝 How to Add a New Competition Function

**When your professor gives you a new optimization function during competition:**

1. **Find the "COMPETITION FUNCTION PLACEHOLDER" cell** (around cell 13)
2. **Replace the placeholder function** with the actual function:
   ```python
   def competition_unknown_function(x):
       """Your function description"""
       # Paste the function implementation here
       return result
   ```
3. **Go to cell 23** (Quick Competition Test) and uncomment/modify:
   ```python
   add_competition_function(
       func=competition_unknown_function,
       func_name="Competition Unknown",
       bounds=np.array([[...]]),  # Set your bounds!
       dim=2,  # Set your dimension!
       global_min=0.0  # If known
   )
   
   competition_test = {
       'Competition Unknown': {
           'function': competition_unknown_function,
           'bounds': np.array([[...]]),  # Same bounds
           'dim': 2,  # Same dimension
           'global_min': 0.0
       }
   }
   
   results = compare_algorithms(...)  # Run comparison
   ```
4. **Run the cell** to test your HW6 algorithm against baselines!

**Alternatively**, you can add it directly to `test_functions` dictionary in cell 15.


In [60]:
# Quick test on ALL 10 benchmark functions to verify HW6 algorithm works
print("="*100)
print("QUICK TEST: HW6 Hybrid CMA-ES on All 10 Benchmark Functions")
print("="*100)

test_results = {}

for func_name, func_info in test_functions.items():
    print(f"\n{'='*100}")
    print(f"Testing on {func_name}...")
    print(f"{'='*100}")
    
    try:
        # Test the core algorithm
        best_f, best_x, history = hw6_hybrid_cma_es(
            objective=func_info['function'],
            bounds=func_info['bounds'],
            dim=func_info['dim'],
            n_generations=50,  # Quick test with fewer generations
            mu=4,
            lam=10,
            seed=42,
        )
        
        # Test the wrapper function
        best_f_wrapper = HW6(
            f=func_info['function'],
            bounds=func_info['bounds'],
            dim=func_info['dim'],
            n_generations=50,
            seed=42
        )
        
        test_results[func_name] = {
            'core_best': best_f,
            'wrapper_best': best_f_wrapper,
            'global_min': func_info['global_min']
        }
        
        print(f"  ✓ Core algorithm best fitness: {best_f:.6e}")
        print(f"  ✓ Wrapper algorithm best fitness: {best_f_wrapper:.6e}")
        print(f"  ✓ Global minimum (reference): {func_info['global_min']}")
        print(f"  ✓ Improvement: {history[0] - history[-1]:.6f}")
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        test_results[func_name] = {'error': str(e)}

print("\n" + "="*100)
print("TEST SUMMARY")
print("="*100)
print(f"{'Function':<20} {'Core Best':<20} {'Wrapper Best':<20} {'Global Min':<15}")
print("-"*100)

for func_name, results in test_results.items():
    if 'error' not in results:
        print(f"{func_name:<20} {results['core_best']:<20.6e} {results['wrapper_best']:<20.6e} {results['global_min']:<15.6e}")
    else:
        print(f"{func_name:<20} {'ERROR':<20} {'ERROR':<20} {'-':<15}")

print("="*100)
print("✅ Quick test complete! Your algorithm has been tested on all 10 benchmark functions.")
print("="*100)



QUICK TEST: HW6 Hybrid CMA-ES on All 10 Benchmark Functions

Testing on Sphere...
  ✓ Core algorithm best fitness: 1.178296e-15
  ✓ Wrapper algorithm best fitness: 7.360495e-13
  ✓ Global minimum (reference): 0.0
  ✓ Improvement: 8.261496

Testing on Rosenbrock...
  ✓ Core algorithm best fitness: 1.187069e-03
  ✓ Wrapper algorithm best fitness: 1.187069e-03
  ✓ Global minimum (reference): 0.0
  ✓ Improvement: 0.079919

Testing on Rastrigin...
  ✓ Core algorithm best fitness: 9.949591e-01
  ✓ Wrapper algorithm best fitness: 1.989918e+00
  ✓ Global minimum (reference): 0.0
  ✓ Improvement: 30.298930

Testing on Ackley...
  ✓ Core algorithm best fitness: 6.729325e-04
  ✓ Wrapper algorithm best fitness: 4.665247e-05
  ✓ Global minimum (reference): 0.0
  ✓ Improvement: 18.572762

Testing on Griewank...
  ✓ Core algorithm best fitness: 1.738934e-01
  ✓ Wrapper algorithm best fitness: 7.831587e-02
  ✓ Global minimum (reference): 0.0
  ✓ Improvement: 29.177901

Testing on Schwefel...
  ✓ Core 